In [1]:
from pymongo import MongoClient
from pymongo.server_api import ServerApi
import os
from dotenv import load_dotenv
import pandas as pd


load_dotenv()

mongo_uri = os.getenv("MONGODB_uri") #pegando a uri para entrar no db

client = MongoClient(mongo_uri, server_api=ServerApi('1'))  #fazendo acesso ao banco de dados

db = client["IBGE"] #acesso a o banco de dadaos guardado nesta variavel

print(db)



Database(MongoClient(host=['ac-a1lzdqq-shard-00-00.0ffmurh.mongodb.net:27017', 'ac-a1lzdqq-shard-00-01.0ffmurh.mongodb.net:27017', 'ac-a1lzdqq-shard-00-02.0ffmurh.mongodb.net:27017'], document_class=dict, tz_aware=False, connect=True, appname='Cluster0', authsource='admin', replicaset='atlas-z057bt-shard-0', tls=True, server_api=<pymongo.server_api.ServerApi object at 0x000001EBCC8B5BE0>), 'IBGE')


In [2]:
data = db["PNADC"] #acesso a colection desejada
print(data)

Collection(Database(MongoClient(host=['ac-a1lzdqq-shard-00-00.0ffmurh.mongodb.net:27017', 'ac-a1lzdqq-shard-00-01.0ffmurh.mongodb.net:27017', 'ac-a1lzdqq-shard-00-02.0ffmurh.mongodb.net:27017'], document_class=dict, tz_aware=False, connect=True, appname='Cluster0', authsource='admin', replicaset='atlas-z057bt-shard-0', tls=True, server_api=<pymongo.server_api.ServerApi object at 0x000001EBCC8B5BE0>), 'IBGE'), 'PNADC')


In [3]:
data.count_documents({}) #Conta quantos documentos tem dentro da collection 

1

In [4]:
#data.find() isso aponta para os dados em memoria entretanto 
#precisamos fazer o cast que no caso e passar para um tipo de variavel que o python entenda 

data = list(data.find()) #Serializacao para conseguirmos manipular estes dados no caso passamos para uma lista pq o python entenda e consguimos sair em diante

In [5]:
serie = data[0]['resultados'][0]["series"][0]['serie']

In [ ]:
df = pd.DataFrame.from_dict(serie, orient='index',columns=['valor'])


,periodo,valor
0,201201,9.6
1,201202,8.3
2,201203,9.4
3,201204,9.2
4,201301,10.7
5,201302,9.7
6,201303,8.5
7,201304,7.4
8,201401,8.8
9,201402,8.0


In [ ]:
df.index.name = "periodo"
df = df.reset_index()
df

In [7]:
df.info()
#Aqui percebe-se que as duas colunas estao erradas ja que eram pra ser data e float mas como tiramos da api ela aparece como str

<class 'pandas.DataFrame'>
RangeIndex: 58 entries, 0 to 57
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   periodo  58 non-null     str  
 1   valor    58 non-null     str  
dtypes: str(2)
memory usage: 1.0 KB


In [8]:
#ent vamos tentar mudar para float

#df = df['valor'].astype(float) 
#nao conseguimos pois os dados ausentes estao como '...' ent teremos que colocar outro valor de acordo com a decisao de negocio pois isso interfere diretamente na media
#e mediana da analise de dados 

df['valor'] = df['valor'].replace("...",'0')

df['valor'] = df['valor'].astype(float)
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 58 entries, 0 to 57
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   periodo  58 non-null     str    
 1   valor    58 non-null     float64
dtypes: float64(1), str(1)
memory usage: 1.0 KB


In [9]:
df['ano'] = df['periodo'].str[:4]
df['tri'] = df['periodo'].str[-2:].astype(int)
df

,periodo,valor,ano,tri
0,201201,9.6,2012,1
1,201202,8.3,2012,2
2,201203,9.4,2012,3
3,201204,9.2,2012,4
4,201301,10.7,2013,1
5,201302,9.7,2013,2
6,201303,8.5,2013,3
7,201304,7.4,2013,4
8,201401,8.8,2014,1
9,201402,8.0,2014,2


In [10]:
df['periodo'] = pd.PeriodIndex(df['ano'] + "Q" + df["tri"].astype(str),freq='Q') 
#Este na concatenacao Q serve para ficar no formato que o pandas trabalha
# e o freq = "Q" e para o pandas entender que estamos trabalhando com trimestres

df

,periodo,valor,ano,tri
0,2012Q1,9.6,2012,1
1,2012Q2,8.3,2012,2
2,2012Q3,9.4,2012,3
3,2012Q4,9.2,2012,4
4,2013Q1,10.7,2013,1
5,2013Q2,9.7,2013,2
6,2013Q3,8.5,2013,3
7,2013Q4,7.4,2013,4
8,2014Q1,8.8,2014,1
9,2014Q2,8.0,2014,2


In [ ]:
df['periodo'] = df['periodo'].dt.to_timestamp() #facilita ainda mais nas nossas vidas pq nos possiblilita usar mais metodos
df

,periodo,valor,ano,tri
0,2012-01-01,9.6,2012,1
1,2012-04-01,8.3,2012,2
2,2012-07-01,9.4,2012,3
3,2012-10-01,9.2,2012,4
4,2013-01-01,10.7,2013,1
5,2013-04-01,9.7,2013,2
6,2013-07-01,8.5,2013,3
7,2013-10-01,7.4,2013,4
8,2014-01-01,8.8,2014,1
9,2014-04-01,8.0,2014,2


In [ ]:
import sqlite3 #banco relacional porem em memoria

conn = sqlite3.connect("ibge.db") #conexao ao banco de dados
df.to_sql("pnadc",conn,if_exists="replace",index=False)
# aqui estamos passando o banco pra SQL onde o nome desta tabela e "pnadc", dps a variavel que representa a conexao e, nao queremos que o index apareca na tabela ent index=False
conn.close()
#Boa pratica sempre fechar a conexao com o banco assim que acabar